# M2-T06: Quality / Handling Signals and Promotion Flag

**Done when (from the Task Board):**
1. `temp_abuse_events`, `handling_score`, `packaging_score` and `supplier_score` are prepared as direct Track 3A inputs (scale and outliers checked).
2. `is_promoted` is carried forward as its own feature, for the M3-T05 confounding check.
3. `spoilage_sensitivity` is included only if it was cleared in M1-T06.
4. Optional: `distribution_hours` enrichment, only if time allows. (`initial_quantity` is handled in M2-T03.)

| Done-when item | Where it is answered |
|---|---|
| 1. Four signals prepared, scale and outliers checked | Sections 2-4 (checks) and Section 5 (signal check) |
| 2. `is_promoted` as its own feature | Section 7 |
| 3. `spoilage_sensitivity` only if cleared | Section 6 |
| 4. Optional `distribution_hours` | Deviations from plan |
| Decisions and hand-off | Section 8 (verdict), Section 9 (paste-ready text), Section 10 (saved files) |

In [1]:
import hashlib
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'config.py').exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.config import RAW_DATASET_PATH, DATA_PROCESSED, TABLES_DIR, REPORTS_M2_DIR, RANDOM_STATE
from src.pipeline import LOG_COLS, SCALE_COLS, BINARY_COLS

np.random.seed(RANDOM_STATE)
print('Project root:', ROOT)

Project root: c:\Users\ASUS\Desktop\NHA-5-150


## 1. Load the raw data

The raw file is only read, never modified. The SHA256 is printed to show every teammate is on the same dataset version.

In [2]:
EXPECTED_SHA256 = 'de94302b867c9debedfd45c431306623fdfc038f5ed8ca17736339b4460a6674'

df = pd.read_csv(RAW_DATASET_PATH)
sha = hashlib.sha256(RAW_DATASET_PATH.read_bytes()).hexdigest()
print('File  :', RAW_DATASET_PATH)
print('SHA256:', sha)
print('Shape :', df.shape)

SIGNALS = ['temp_abuse_events', 'handling_score', 'packaging_score', 'supplier_score']
EXTRA = ['spoilage_sensitivity', 'is_promoted']
needed = SIGNALS + EXTRA + ['was_spoiled', 'markdown_applied', 'transaction_date', 'category', 'record_id']

assert sha == EXPECTED_SHA256, 'Different dataset version'
assert df.shape == (100_000, 42)
assert not set(needed) - set(df.columns)
assert df[needed].notna().all().all(), 'Unexpected nulls'
assert df['record_id'].is_unique

File  : C:\Users\ASUS\Desktop\NHA-5-150\data\raw\perishable_goods_management.csv
SHA256: de94302b867c9debedfd45c431306623fdfc038f5ed8ca17736339b4460a6674
Shape : (100000, 42)


## 2. How the team pipeline already treats these columns

M1-T04 defined the preprocessing in `src/pipeline.py`. Reading the roles from there (instead of re-defining them) keeps this task consistent with the rest of the project.

In [3]:
roles = {}
for c in SIGNALS + EXTRA:
    if c in LOG_COLS:
        roles[c] = 'log1p + StandardScaler'
    elif c in SCALE_COLS:
        roles[c] = 'StandardScaler'
    elif c in BINARY_COLS:
        roles[c] = 'binary passthrough'
    else:
        roles[c] = 'NOT IN PIPELINE'
roles = pd.Series(roles, name='pipeline_treatment').to_frame()
assert 'NOT IN PIPELINE' not in roles['pipeline_treatment'].values
roles

,pipeline_treatment
temp_abuse_events,log1p + StandardScaler
handling_score,StandardScaler
packaging_score,StandardScaler
supplier_score,StandardScaler
spoilage_sensitivity,StandardScaler
is_promoted,binary passthrough


**Conclusion:** all six columns are already part of the M1-T04 pipeline. `temp_abuse_events` gets `log1p` then standard scaling, the other three signals and `spoilage_sensitivity` get standard scaling, and `is_promoted` is passed through as a 0/1 flag. This task checks that this treatment is appropriate and packages the columns for Track 3A.

## 3. Scale and outlier check

Outliers are counted with the IQR rule (1.5 x IQR beyond the quartiles).

In [4]:
rows = []
for c in SIGNALS + ['spoilage_sensitivity']:
    s = df[c]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100
    rows.append({'column': c, 'min': s.min(), 'max': s.max(), 'mean': round(s.mean(), 3),
                 'std': round(s.std(), 3), 'distinct_values': s.nunique(),
                 'skew': round(s.skew(), 3), 'iqr_outliers_pct': round(outliers, 1)})
scale_check = pd.DataFrame(rows).set_index('column')
scale_check

,min,max,mean,std,distinct_values,skew,iqr_outliers_pct
column,,,,,,,
temp_abuse_events,0.0,10.00,0.801,1.079,11,1.666,7.8
handling_score,4.0,10.00,6.999,2.000,7,0.001,0.0
packaging_score,5.0,10.00,7.500,1.704,6,0.003,0.0
supplier_score,6.0,10.00,8.208,1.398,5,-0.367,0.0
spoilage_sensitivity,0.3,0.95,0.686,0.215,9,-0.443,0.0


In [5]:
print('temp_abuse_events value counts:')
print(df['temp_abuse_events'].value_counts().sort_index().to_dict())
print('Skew before log1p:', round(df['temp_abuse_events'].skew(), 3), '| after:', round(np.log1p(df['temp_abuse_events']).skew(), 3))

temp_abuse_events value counts:
{0: 52161, 1: 27516, 2: 12485, 3: 5017, 4: 1903, 5: 624, 6: 211, 7: 64, 8: 16, 9: 2, 10: 1}
Skew before log1p: 1.666 | after: 0.679


**Conclusion:**
- `handling_score` (4-10), `packaging_score` (5-10) and `supplier_score` (6-10) are small ordinal scales with no IQR outliers and skew close to zero, so standard scaling is enough.
- `temp_abuse_events` is a right-skewed count (skew 1.67, 7.8% IQR outliers). The large values are valid counts (0-10 events), not data errors, so they are **kept**. `log1p` reduces the skew to 0.68, which is why M1-T04 applies it before scaling.
- `spoilage_sensitivity` has 10 categories sharing 9 distinct values and no outliers.

## 4. Scaling without leakage

Scalers are fitted on the training period only (time-ordered split from M1-T04: train up to 2024-08-07) and then applied to the test period. The same steps as `src/pipeline.py` are used (`log1p` only for `temp_abuse_events`).

In [6]:
TRAIN_END = pd.Timestamp('2024-08-07')
dates = pd.to_datetime(df['transaction_date'])
is_train = dates <= TRAIN_END
assert int(is_train.sum()) == 80_042 and int((~is_train).sum()) == 19_958

def make_scaler(col):
    steps = []
    if col in LOG_COLS:
        steps.append(('log1p', FunctionTransformer(np.log1p)))
    steps.append(('scale', StandardScaler()))
    return Pipeline(steps)

SCALED = SIGNALS + ['spoilage_sensitivity']
z = pd.DataFrame(index=df.index)
for c in SCALED:
    scaler = make_scaler(c).fit(df.loc[is_train, [c]])
    z[c + '_z'] = scaler.transform(df[[c]]).ravel()

scaling_check = pd.DataFrame({
    'train_mean': z[is_train].mean().round(6),
    'train_std': z[is_train].std(ddof=0).round(6),
    'test_mean': z[~is_train].mean().round(3),
    'test_min': z[~is_train].min().round(2),
    'test_max': z[~is_train].max().round(2),
})
assert z[is_train].mean().abs().max() < 1e-9
assert (z[is_train].std(ddof=0) - 1).abs().max() < 1e-9
scaling_check

,train_mean,train_std,test_mean,test_min,test_max
temp_abuse_events_z,0.0,1.0,0.002,-0.87,3.19
handling_score_z,-0.0,1.0,-0.005,-1.50,1.50
packaging_score_z,0.0,1.0,-0.004,-1.47,1.47
supplier_score_z,-0.0,1.0,0.002,-1.58,1.28
spoilage_sensitivity_z,0.0,1.0,-0.003,-1.79,1.23


**Conclusion:** on the training period every scaled column has mean 0 and standard deviation 1. The test period lands close to 0 as well, so there is no sign of drift between the two periods for these columns.

## 5. Do the signals relate to spoilage?

Point-biserial correlation with `was_spoiled`. A Bonferroni threshold over the six columns (0.05 / 6) is used so that the large sample does not make everything look significant.

In [7]:
ALL6 = SIGNALS + EXTRA
ALPHA = 0.05 / len(ALL6)
rows = []
for c in ALL6:
    r, p = stats.pearsonr(df[c], df['was_spoiled'])
    rows.append({'column': c, 'pearson_r': round(r, 4), 'p_value': p, 'significant': bool(p < ALPHA)})
signal = pd.DataFrame(rows).set_index('column')
signal

,pearson_r,p_value,significant
column,,,
temp_abuse_events,0.0287,1.215015e-19,True
handling_score,-0.0563,4.911484e-71,True
packaging_score,-0.0885,6.199492e-173,True
supplier_score,0.0029,3.581011e-01,False
spoilage_sensitivity,0.0370,1.447920e-31,True
is_promoted,0.0019,5.515959e-01,False


In [8]:
by_pkg = df.groupby('packaging_score')['was_spoiled'].mean().round(3).to_frame('spoilage_rate').T
by_hnd = df.groupby('handling_score')['was_spoiled'].mean().round(3).to_frame('spoilage_rate').T
by_sup = df.groupby('supplier_score')['was_spoiled'].mean().round(3).to_frame('spoilage_rate').T
print('packaging_score'); print(by_pkg.to_string())
print('handling_score'); print(by_hnd.to_string())
print('supplier_score'); print(by_sup.to_string())

packaging_score
packaging_score     5      6      7     8      9      10
spoilage_rate    0.246  0.231  0.199  0.18  0.166  0.145
handling_score
handling_score     4      5      6      7      8     9      10
spoilage_rate   0.228  0.217  0.202  0.196  0.186  0.17  0.161
supplier_score
supplier_score     6      7      8      9      10
spoilage_rate   0.193  0.194  0.194  0.195  0.197


**Conclusion:**
- `packaging_score` (r = -0.0885) and `handling_score` (r = -0.0563) are the clearest signals: spoilage falls from 24.6% to 14.5% as packaging goes from 5 to 10, and from 22.8% to 16.1% as handling goes from 4 to 10.
- `temp_abuse_events` (r = +0.0287) is a small but significant risk driver.
- `supplier_score` (r = +0.0029, p = 0.36) shows **no signal**: spoilage stays between 19.3% and 19.7% at every score level. This matches the M2-T01 finding (`supplier_score` is a null factor).

## 6. `spoilage_sensitivity`: cleared in M1-T06, but redundant with `category`

M1-T06 cleared it as a legitimate input (no leakage). The check below shows why M2-T01 still recommends not using it next to `category`.

In [9]:
const_within_category = bool((df.groupby('category')['spoilage_sensitivity'].nunique() == 1).all())
fitted = df.groupby('category')['spoilage_sensitivity'].transform('mean')
resid = ((df['spoilage_sensitivity'] - fitted) ** 2).sum()
total = ((df['spoilage_sensitivity'] - df['spoilage_sensitivity'].mean()) ** 2).sum()
r2_category = 1 - resid / total
print('Constant within every category:', const_within_category)
print('Distinct values               :', df['spoilage_sensitivity'].nunique())
print('R2 explained by category      :', round(r2_category, 4))
df.groupby('category')['spoilage_sensitivity'].first().sort_values().to_frame('spoilage_sensitivity').T

Constant within every category: True
Distinct values               : 9
R2 explained by category      : 1.0


category,Frozen_Meals,Beverages,Bakery,Produce,Dairy,Deli,Pharmaceuticals,Meat,Ready_to_Eat,Seafood
spoilage_sensitivity,0.3,0.4,0.5,0.6,0.7,0.75,0.85,0.9,0.9,0.95


**Conclusion:** `spoilage_sensitivity` is a fixed attribute of the product category (R² = 1.0). It is not leakage, so it is included in the output as the task allows, but it adds no information once `category` is in the model. It is flagged as redundant in the decision table.

## 7. `is_promoted` as its own feature

The M3-T05 confounding check needs to separate promotional discounts from spoilage-driven ones, so `is_promoted` is kept as a separate 0/1 column and is **not** merged into the discount flags.

In [10]:
print(pd.crosstab(df['is_promoted'], df['markdown_applied'], rownames=['is_promoted'], colnames=['markdown_applied']))
corr_promo_md = df['is_promoted'].corr(df['markdown_applied'])
print('\nShare promoted               :', round(df['is_promoted'].mean(), 3))
print('corr(is_promoted, markdown)  :', round(corr_promo_md, 4))
print('Spoilage rate by is_promoted :', df.groupby('is_promoted')['was_spoiled'].mean().round(4).to_dict())
assert set(df['is_promoted'].unique()) == {0, 1}

markdown_applied      0      1
is_promoted                   
0                 58984  26118
1                 10365   4533

Share promoted               : 0.149
corr(is_promoted, markdown)  : -0.002
Spoilage rate by is_promoted : {0: 0.1941, 1: 0.1962}


**Conclusion:** promoted and non-promoted batches both appear with and without a markdown, and the correlation between the two is -0.002. So `is_promoted` carries information that the discount flags do not, which is what the M3-T05 check needs. On its own it has almost no relation with spoilage (19.4% vs 19.6%), which is why M2-T01 drops it for Track 3A and keeps it for Track 3B as a confounder.

## 8. Verdict (decided by the evidence above)

In [11]:
decision = signal[['pearson_r', 'p_value', 'significant']].copy()
decision['pipeline_treatment'] = roles['pipeline_treatment']

def track_3a_note(c):
    if c == 'is_promoted':
        return 'own feature for M3-T05; null for spoilage, so not a Track 3A driver (M2-T01)'
    if c == 'spoilage_sensitivity':
        return 'cleared by M1-T06; redundant with category (R2 = 1.0), M2-T01 recommends drop'
    if decision.loc[c, 'significant']:
        return 'keep as Track 3A input'
    return 'null signal; M2-T01 recommends drop'

decision['note'] = [track_3a_note(c) for c in decision.index]

assert round(signal.loc['packaging_score', 'pearson_r'], 4) == -0.0885
assert round(signal.loc['handling_score', 'pearson_r'], 4) == -0.0563
assert round(signal.loc['temp_abuse_events', 'pearson_r'], 4) == 0.0287
assert round(signal.loc['supplier_score', 'pearson_r'], 4) == 0.0029
assert signal.loc[['packaging_score', 'handling_score', 'temp_abuse_events', 'spoilage_sensitivity'], 'significant'].all()
assert not signal.loc[['supplier_score', 'is_promoted'], 'significant'].any()
assert const_within_category and round(r2_category, 4) == 1.0
assert round(df.loc[df['packaging_score'] == 5, 'was_spoiled'].mean(), 3) == 0.246
assert round(df.loc[df['packaging_score'] == 10, 'was_spoiled'].mean(), 3) == 0.145
assert round(df['temp_abuse_events'].skew(), 2) == 1.67
assert round(corr_promo_md, 3) == -0.002

decision[['pearson_r', 'significant', 'pipeline_treatment', 'note']]

,pearson_r,significant,pipeline_treatment,note
column,,,,
temp_abuse_events,0.0287,True,log1p + StandardScaler,keep as Track 3A input
handling_score,-0.0563,True,StandardScaler,keep as Track 3A input
packaging_score,-0.0885,True,StandardScaler,keep as Track 3A input
supplier_score,0.0029,False,StandardScaler,null signal; M2-T01 recommends drop
spoilage_sensitivity,0.0370,True,StandardScaler,cleared by M1-T06; redundant with category (R2...
is_promoted,0.0019,False,binary passthrough,"own feature for M3-T05; null for spoilage, so ..."


## 9. Resolution for the report (paste-ready)

> **Quality / handling signals and promotion flag (M2-T06)**
>
> **Prepared inputs.** `temp_abuse_events`, `handling_score`, `packaging_score` and `supplier_score` were checked for scale and outliers and are ready as Track 3A inputs. `handling_score`, `packaging_score` and `supplier_score` are bounded ordinal scales (4-10, 5-10, 6-10) with no IQR outliers. `temp_abuse_events` is a right-skewed count (skew 1.67, 7.8% IQR outliers, values 0-10); the large values are valid and kept, and `log1p` reduces the skew to 0.68. Scaling follows the M1-T04 pipeline and is fitted on the training period only (up to 2024-08-07), so no test information leaks into the scaler.
>
> **Signal.** `packaging_score` (r = -0.0885) and `handling_score` (r = -0.0563) are protective: spoilage falls from 24.6% to 14.5% as packaging improves from 5 to 10. `temp_abuse_events` is a small risk driver (r = +0.0287). `supplier_score` shows no signal (r = +0.0029, p = 0.36).
>
> **`is_promoted`.** Carried forward as its own 0/1 feature, separate from the discount flags (correlation with `markdown_applied` = -0.002), so the M3-T05 confounding check can separate promotional discounts from spoilage-driven ones.
>
> **`spoilage_sensitivity`.** Cleared in M1-T06 and included in the output. It is constant within each category (R² = 1.0), so it is redundant with `category`.
>
> **Recommendation.** Use `packaging_score`, `handling_score` and `temp_abuse_events` in Track 3A. `supplier_score` (null signal) and `spoilage_sensitivity` (redundant with category) are prepared as the task requires but are not recommended for the final feature set, in line with M2-T01.

## 10. Save the outputs

The feature file goes to `data/processed/` (not committed). The check table goes to `reports/tables/` and the decision record to `reports/M2/`.

In [12]:
RAW_KEEP = SIGNALS + EXTRA
out = pd.concat([
    df[['record_id']],
    is_train.map({True: 'train', False: 'test'}).rename('split'),
    df[RAW_KEEP],
    z,
], axis=1)

forbidden = {'spoilage_risk', 'units_sold', 'units_wasted', 'waste_pct', 'revenue', 'waste_cost',
             'profit', 'profit_margin_pct', 'discount_pct', 'markdown_applied', 'selling_price', 'was_spoiled'}
assert not forbidden & set(out.columns)
assert len(out) == len(df) and out['record_id'].is_unique
assert out.isna().sum().sum() == 0
assert out['handling_score'].between(4, 10).all()
assert out['packaging_score'].between(5, 10).all()
assert out['supplier_score'].between(6, 10).all()
assert out['temp_abuse_events'].between(0, 10).all()

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_M2_DIR.mkdir(parents=True, exist_ok=True)

out.to_csv(DATA_PROCESSED / 'M2_T06_quality_handling_features.csv', index=False)
decision.join(scale_check, how='left').to_csv(TABLES_DIR / 'M2_T06_feature_checks.csv')

resolution = {
    'task': 'M2-T06',
    'dataset_sha256': sha,
    'rows': int(len(df)),
    'train_rows': int(is_train.sum()),
    'test_rows': int((~is_train).sum()),
    'columns_in_output': list(out.columns),
    'pearson_r_with_was_spoiled': {c: float(signal.loc[c, 'pearson_r']) for c in signal.index},
    'track_3a_notes': {c: decision.loc[c, 'note'] for c in decision.index},
    'spoilage_sensitivity_r2_category': round(float(r2_category), 4),
    'is_promoted_corr_with_markdown_applied': round(float(corr_promo_md), 4),
    'distribution_hours_enrichment_tried': False,
}
with open(REPORTS_M2_DIR / 'm2_t06_resolution.json', 'w') as f:
    json.dump(resolution, f, indent=2)

print('All checks passed')
print(out.shape)
out.head()

All checks passed
(100000, 13)


,record_id,split,temp_abuse_events,handling_score,packaging_score,supplier_score,spoilage_sensitivity,is_promoted,temp_abuse_events_z,handling_score_z,packaging_score_z,supplier_score_z,spoilage_sensitivity_z
0,1,test,0,9,10,9,0.50,0,-0.86939,0.998974,1.465142,0.567088,-0.863086
1,2,train,0,8,8,6,0.90,0,-0.86939,0.499374,0.292243,-1.578611,0.993501
2,3,test,0,4,8,9,0.50,0,-0.86939,-1.499022,0.292243,0.567088,-0.863086
3,4,train,0,5,9,6,0.85,0,-0.86939,-0.999423,0.878693,-1.578611,0.761428
4,5,train,0,5,6,6,0.90,0,-0.86939,-0.999423,-0.880656,-1.578611,0.993501


## Deviations from plan

1. **`supplier_score`** was prepared as the task requires, but it shows no signal (r = 0.003, p = 0.36) and M2-T01 already marks it as a drop. Recommendation: do not use it in the final Track 3A feature set.
2. **`spoilage_sensitivity`** was included because M1-T06 cleared it. It is redundant with `category` (R² = 1.0), and M2-T01 recommends dropping it. The team should decide whether to keep it. Also, `feature_dictionary.csv` still lists it as `hold until M1-T06`, which should be updated to the M1-T06 verdict (`allowed`).
3. **`distribution_hours`** enrichment of the sell-through risk factor (M2-T03) was not tried; it was optional. `distribution_hours` is already scaled in the M1-T04 pipeline and kept by M2-T01 as a Track 3A driver, so nothing is missing for Track 3A.